# Data Leakage Audit

 **Objective**: Ensure that the machine learning models are not cheating by exploiting data leakage.
 This includes checking for overlapping captures between splits, exact duplicate flows across
 train/test, and ensuring no prohibited metadata (like IPs, ports, or timestamps) has leaked
 into the feature space.

In [1]:
%matplotlib inline

import pandas as pd
import numpy as np

from src.pipeline.data_preparation import load_and_prepare_data
from src.pipeline.feature_pipeline import FeaturePipeline
from src.utils.logging import setup_logger

logger = setup_logger(level="INFO")

In [2]:
logger.info("Loading datasets...")
df_all = load_and_prepare_data()

df_train = df_all[df_all["split"] == "train"].copy()
df_val = df_all[df_all["split"] == "val"].copy()
df_test = df_all[df_all["split"] == "test"].copy()

2026-03-29 13:00:08 | INFO | ai-vpn-firewall | Loading datasets...
2026-03-29 13:00:08 | INFO | ai-vpn-firewall | Loading VNAT (PCAP-based)...
2026-03-29 13:00:08 | INFO | ai-vpn-firewall | Loading ISCX (PCAP-based)...
2026-03-29 13:00:08 | INFO | ai-vpn-firewall | Loading USBVPN (JSON-based)...
2026-03-29 13:00:08 | INFO | ai-vpn-firewall | Removing exact duplicate flows across feature columns...
2026-03-29 13:00:08 | INFO | ai-vpn-firewall | Removed 488 duplicate flows (0.70%)
2026-03-29 13:00:08 | INFO | ai-vpn-firewall | Ensuring numeric dtypes for feature columns...
2026-03-29 13:00:08 | INFO | ai-vpn-firewall | ✓ All feature columns successfully converted to numeric dtypes
2026-03-29 13:00:08 | INFO | ai-vpn-firewall | Metadata columns present for analysis only: ['source_capture_id', 'source_file']
2026-03-29 13:00:08 | INFO | ai-vpn-firewall | Multi-Domain Pool Created: (69070, 38)
2026-03-29 13:00:08 | INFO | ai-vpn-firewall | Datasets: {'usbvpn': 49177, 'iscx': 11789, 'vnat': 

## 1. Group Leakage: Capture ID Overlap

In network traffic classification, splitting by `capture_id` (or PCAP file) is critical.
Random splitting allows packets from the same session or background noise to exist in both
Train and Test, causing massive leakage.

In [3]:
logger.info("Running Group Leakage Check...")

train_caps = set(df_train["capture_id"].unique())
val_caps = set(df_val["capture_id"].unique())
test_caps = set(df_test["capture_id"].unique())

2026-03-29 13:00:09 | INFO | ai-vpn-firewall | Running Group Leakage Check...


In [4]:
train_test_overlap = train_caps.intersection(test_caps)
train_val_overlap = train_caps.intersection(val_caps)
val_test_overlap = val_caps.intersection(test_caps)

In [5]:
print("--- Capture ID Overlap Results ---")
print(f"Train / Test Overlap: {len(train_test_overlap)} captures")
if train_test_overlap:
    print(f"  -> Leakage Detected! Overlapping captures: {train_test_overlap}")

print(f"Train / Val Overlap:  {len(train_val_overlap)} captures")
if train_val_overlap:
    print(f"  -> Leakage Detected! Overlapping captures: {train_val_overlap}")

print(f"Val / Test Overlap:   {len(val_test_overlap)} captures")
if val_test_overlap:
    print(f"  -> Leakage Detected! Overlapping captures: {val_test_overlap}")

if not train_test_overlap and not train_val_overlap and not val_test_overlap:
    print("\nSUCCESS: No capture groups leak across splits. The dataset is strictly grouped.")

--- Capture ID Overlap Results ---
Train / Test Overlap: 0 captures
Train / Val Overlap:  0 captures
Val / Test Overlap:   0 captures

SUCCESS: No capture groups leak across splits. The dataset is strictly grouped.


## 2. Feature Schema Metadata Leakage

Ensure that no metadata features (ports, IPs, connection strings, timestamps)
accidentally made it into the `FeaturePipeline` model input.

In [6]:
logger.info("Running Feature Schema Metadata Leakage Check...")

pipe = FeaturePipeline().fit(df_train)
model_features = pipe.model_feature_names()

prohibited_keywords = [
    "ip", "port", "mac", "time", "stamp",
    "id", "capture", "file", "split", "dataset", "source_capture"
]

leaked_features = []
for f in model_features:
    for kw in prohibited_keywords:
        if kw in f.lower():
            leaked_features.append(f)
            break

print("--- Metadata Leakage Results ---")
if leaked_features:
    raise ValueError(f"Metadata leakage detected in model feature space: {leaked_features}")
else:
    print("SUCCESS: No metadata keywords found in the model feature space.")

2026-03-29 13:00:09 | INFO | ai-vpn-firewall | Running Feature Schema Metadata Leakage Check...
--- Metadata Leakage Results ---
SUCCESS: No metadata keywords found in the model feature space.


## 3. Exact Duplicate Flows Across Splits

If identical flow behaviors are in both Train and Test, it inflates test performance.

In [7]:
logger.info("Running Duplicate Flow Check...")

# We check duplicates based only on the core behavioral features
# (ignoring IDs and labels)
X_train_feats = pipe.transform(df_train)[model_features]
X_test_feats = pipe.transform(df_test)[model_features]

2026-03-29 13:00:09 | INFO | ai-vpn-firewall | Running Duplicate Flow Check...


In [8]:
# Find UNIQUE identical flows (not Cartesian product from merge)
train_tuples = set(map(tuple, X_train_feats.values))
test_tuples = set(map(tuple, X_test_feats.values))

# Count test samples that are duplicates of train (including multiplicity)
test_values = X_test_feats.values
test_duplicates = [tuple(row) in train_tuples for row in test_values]
duplicates_count = sum(test_duplicates)

In [9]:
print("--- Exact Duplicate Flows Check ---")
print(f"Identical flows existing in both Train and Test: {duplicates_count}")

--- Exact Duplicate Flows Check ---
Identical flows existing in both Train and Test: 15


In [10]:
# Calculate percentage of test set that is leaked
if len(X_test_feats) > 0:
    leak_pct = (duplicates_count / len(X_test_feats)) * 100
    print(f"Percentage of Test Set leaking from Train: {leak_pct:.2f}%")

    if leak_pct > 5.0:
        print("WARNING: High duplicate flow rate. Model might be memorizing.")
    else:
        print("SUCCESS: Duplicate flow rate is within acceptable bounds for network traffic.")

Percentage of Test Set leaking from Train: 0.16%
SUCCESS: Duplicate flow rate is within acceptable bounds for network traffic.


## 3.5 Cross-Split Deduplication (CORRECTIVE)

Apply the FeaturePipeline deduplication to remove cross-split duplicates and verify the fix.

In [11]:
logger.info("Applying cross-split deduplication fix...")

df_all_combined = pd.concat([
    pipe.transform(df_train),
    pipe.transform(df_test)
], ignore_index=False)

df_deduped, num_removed = pipe.remove_cross_split_duplicates_in_transformed_space(
    df_all_combined,
    model_features,
    train_split="train",
    val_split="val",
    test_split="test"
)

2026-03-29 13:00:10 | INFO | ai-vpn-firewall | Applying cross-split deduplication fix...
2026-03-29 13:00:11 | INFO | ai-vpn-firewall | Created reference set of 51083 unique train feature vectors
2026-03-29 13:00:11 | INFO | ai-vpn-firewall | Marked 15 duplicate flows from test set (matching train)
2026-03-29 13:00:11 | WARNING | ai-vpn-firewall | Cross-split deduplication removed 15 flows (0.02%): {'test': np.int64(15)}


In [12]:
print("--- After Cross-Split Deduplication ---")
print(f"Flows removed from test set: {num_removed}")
print(f"Original test set size: {len(df_test)}")
print(f"New test set size: {len(df_deduped[df_deduped['split'] == 'test'])}")

# Re-check for duplicates after deduplication
df_train_deduped = df_deduped[df_deduped["split"] == "train"]
df_test_deduped = df_deduped[df_deduped["split"] == "test"]

X_train_deduped = df_train_deduped[model_features]
X_test_deduped = df_test_deduped[model_features]

train_tuples_deduped = set(map(tuple, X_train_deduped.values))
test_values_deduped = X_test_deduped.values
test_duplicates_deduped = [tuple(row) in train_tuples_deduped for row in test_values_deduped]
duplicates_after = sum(test_duplicates_deduped)

print(f"\nIdentical flows after deduplication: {duplicates_after}")

if len(X_test_deduped) > 0:
    leak_pct_after = (duplicates_after / len(X_test_deduped)) * 100
    print(f"Percentage of Test Set leaking after fix: {leak_pct_after:.2f}%")

    if leak_pct_after < 0.1:
        print("SUCCESS: Cross-split duplicates have been effectively removed!")
    else:
        print(f"INFO: {leak_pct_after:.2f}% leakage remains")

--- After Cross-Split Deduplication ---
Flows removed from test set: 15
Original test set size: 9310
New test set size: 9295

Identical flows after deduplication: 0
Percentage of Test Set leaking after fix: 0.00%
SUCCESS: Cross-split duplicates have been effectively removed!


## 4. High Target Correlation (Target Leakage)

Check if any single feature is almost perfectly correlated with the target label.
If a single feature has an absolute correlation > 0.90, it might be a spurious artifact
rather than a true behavioral indicator.

In [13]:
logger.info("Running Target Correlation Check...")

train_corr_df = X_train_feats.copy()
train_corr_df["LABEL_TARGET"] = df_train["label"].astype(int).values

2026-03-29 13:00:12 | INFO | ai-vpn-firewall | Running Target Correlation Check...


In [14]:
correlations = (
    train_corr_df.corr()["LABEL_TARGET"]
    .drop("LABEL_TARGET")
    .abs()
    .sort_values(ascending=False)
)

In [15]:
print("--- Top 5 Features Correlated with Target ---")
print(correlations.head(5))

--- Top 5 Features Correlated with Target ---
sz_all_median    0.240065
sz_all_mean      0.173544
iat_all_mean     0.121184
iat_mean_min     0.120895
iat_std_max      0.118507
Name: LABEL_TARGET, dtype: float64


In [16]:
highly_correlated = correlations[correlations > 0.90]

if not highly_correlated.empty:
    print(
        f"\nWARNING: Found {len(highly_correlated)} features with >0.90 correlation to the label. Investigate for target leakage.")
    print(highly_correlated)
else:
    print("\nSUCCESS: No single feature perfectly predicts the target (max correlation < 0.90).")


SUCCESS: No single feature perfectly predicts the target (max correlation < 0.90).
